# QCM/QMS timestamp–recipe 통합

세 CSV를 읽어 `Mod8/do1`, `Mod8/do2`의 실제 밸브 신호를 기준으로 각 timestamp에 recipe/phase를 붙이고, 하나의 master table을 생성합니다.

- `do1 = 1`: TMA dose (약 1초)
- `do2 = 1`: H2O dose (약 2초)
- 두 신호가 0: N2 purge
- execution table은 pulse 순서, recipe step, 명목 시간 검증에 사용합니다.

In [ ]:
import io
import re
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 180)

## 1. CSV 업로드
아래 셀을 실행하고 세 파일을 한 번에 선택하세요. 런타임에 이미 파일이 있으면 업로드를 건너뜁니다.

In [ ]:
EXPECTED = {
    'datalog': 'datalog_fixed.csv',
    'execution': 'ALD_TMA_H2O_execution_table.csv',
    'pressure': 'ALD_TMA_H2O_pressure_trace.csv',
}

missing = [name for name in EXPECTED.values() if not Path(name).exists()]
if missing:
    from google.colab import files
    print('세 CSV를 선택하세요:', missing)
    uploaded = files.upload()

still_missing = [name for name in EXPECTED.values() if not Path(name).exists()]
if still_missing:
    raise FileNotFoundError(f'파일을 찾을 수 없습니다: {still_missing}')

print('입력 파일 확인 완료')

## 2. 파일 읽기 및 컬럼명 정리

In [ ]:
def read_csv_flexible(path):
    # 구분자를 자동 인식하고, 일반적인 인코딩을 순서대로 시도
    last_error = None
    for enc in ('utf-8-sig', 'utf-8', 'cp949', 'latin1'):
        try:
            return pd.read_csv(path, sep=None, engine='python', encoding=enc)
        except Exception as exc:
            last_error = exc
    raise last_error

def clean_column_name(col):
    col = str(col).strip()
    col = col.replace('ng/cm��', 'ng/cm²').replace('��C', '°C')
    return re.sub(r'\s+', ' ', col)

datalog = read_csv_flexible(EXPECTED['datalog'])
execution = read_csv_flexible(EXPECTED['execution'])
pressure = read_csv_flexible(EXPECTED['pressure'])

for frame in (datalog, execution, pressure):
    frame.columns = [clean_column_name(c) for c in frame.columns]

print('행 수:', {'datalog': len(datalog), 'execution': len(execution), 'pressure': len(pressure)})
display(datalog.head(3))
display(execution.head(8))

## 3. 중복 로그 검증
`datalog`를 master로 사용합니다. pressure trace와 timestamp가 같고 공통 컬럼 값도 같으면 중복 컬럼은 추가하지 않습니다. pressure trace에만 존재하면서 실제 값이 있는 컬럼만 master에 추가합니다.

In [ ]:
TIME_COL = 'Timestamp'
ELAPSED_COL = 'Elapsed Time [s]'
DO1_COL = 'Mod8/do1'
DO2_COL = 'Mod8/do2'

for required in (TIME_COL, ELAPSED_COL, DO1_COL, DO2_COL):
    if required not in datalog.columns:
        raise KeyError(f'datalog에 필수 컬럼이 없습니다: {required}')

datalog[TIME_COL] = pd.to_datetime(datalog[TIME_COL], errors='coerce')
pressure[TIME_COL] = pd.to_datetime(pressure[TIME_COL], errors='coerce')
datalog[ELAPSED_COL] = pd.to_numeric(datalog[ELAPSED_COL], errors='coerce')
pressure[ELAPSED_COL] = pd.to_numeric(pressure[ELAPSED_COL], errors='coerce')

if datalog[TIME_COL].isna().any():
    raise ValueError('datalog Timestamp 변환에 실패한 행이 있습니다.')

same_length = len(datalog) == len(pressure)
same_timestamps = same_length and datalog[TIME_COL].equals(pressure[TIME_COL])
print(f'행 수 동일: {same_length}, timestamp 전체 동일: {same_timestamps}')

common = [c for c in datalog.columns if c in pressure.columns and c not in (TIME_COL,)]
comparison = []
if same_timestamps:
    for c in common:
        left = pd.to_numeric(datalog[c], errors='coerce')
        right = pd.to_numeric(pressure[c], errors='coerce')
        comparable = left.notna() & right.notna()
        max_diff = (left[comparable] - right[comparable]).abs().max() if comparable.any() else np.nan
        comparison.append({'column': c, 'comparable_rows': int(comparable.sum()), 'max_abs_diff': max_diff})
display(pd.DataFrame(comparison))

master = datalog.copy()
pressure_unique = [c for c in pressure.columns if c not in master.columns and pressure[c].notna().any()]
if pressure_unique:
    if not same_timestamps:
        raise ValueError('pressure trace의 고유 컬럼을 붙이려면 timestamp 정렬 규칙을 확인해야 합니다.')
    for c in pressure_unique:
        master[c] = pressure[c].to_numpy()
print('pressure trace에서 추가된 유효 고유 컬럼:', pressure_unique or '없음')

## 4. 실제 digital output pulse 검출 및 recipe 검증

In [ ]:
master[DO1_COL] = pd.to_numeric(master[DO1_COL], errors='coerce').fillna(0).astype(int)
master[DO2_COL] = pd.to_numeric(master[DO2_COL], errors='coerce').fillna(0).astype(int)
if ((master[DO1_COL] == 1) & (master[DO2_COL] == 1)).any():
    raise ValueError('do1과 do2가 동시에 ON인 timestamp가 있습니다.')

master['dose_signal'] = np.select(
    [master[DO1_COL].eq(1), master[DO2_COL].eq(1)],
    ['TMA', 'H2O'],
    default='None'
)
master['pulse_start'] = (master['dose_signal'] != 'None') & (master['dose_signal'].shift(fill_value='None') != master['dose_signal'])
master['pulse_id'] = master['pulse_start'].cumsum().where(master['dose_signal'] != 'None')

pulse_rows = master.loc[master['pulse_start'], [TIME_COL, ELAPSED_COL, 'dose_signal']].copy()
pulse_rows['pulse_sequence'] = np.arange(1, len(pulse_rows) + 1)
pulse_rows = pulse_rows.rename(columns={TIME_COL: 'pulse_start_timestamp', ELAPSED_COL: 'pulse_start_elapsed_s', 'dose_signal': 'detected_precursor'})

execution['Recipe Step'] = pd.to_numeric(execution['Recipe Step'], errors='raise').astype(int)
execution['Dose Time [s]'] = pd.to_numeric(execution['Dose Time [s]'], errors='coerce')
execution['Purge Time [s]'] = pd.to_numeric(execution['Purge Time [s]'], errors='coerce')
recipe_pulses = execution.loc[execution['Precursor'].isin(['TMA', 'H2O'])].copy().reset_index(drop=True)

print('검출 pulse:', pulse_rows['detected_precursor'].value_counts().to_dict())
print('recipe pulse:', recipe_pulses['Precursor'].value_counts().to_dict())
if len(pulse_rows) != len(recipe_pulses):
    raise ValueError(f'pulse 개수 불일치: detected={len(pulse_rows)}, recipe={len(recipe_pulses)}')

pulse_map = pulse_rows.reset_index(names='master_index').join(
    recipe_pulses[['Recipe Step', 'Precursor', 'Dose Time [s]', 'Purge Time [s]', 'ALD Process Type']]
)
pulse_map['precursor_match'] = pulse_map['detected_precursor'].eq(pulse_map['Precursor'])
if not pulse_map['precursor_match'].all():
    display(pulse_map.loc[~pulse_map['precursor_match']])
    raise ValueError('검출된 do 신호 순서와 execution table precursor 순서가 다릅니다.')
display(pulse_map.head(10))
print('모든 pulse의 precursor 순서가 recipe와 일치합니다.')

## 5. 각 timestamp에 recipe step과 phase 대응
pulse 시작은 digital output으로 고정합니다. 독립 purge(`Precursor=None`)는 execution table의 위치와 명목 시간을 이용해 구분합니다.

In [ ]:
master['recipe_step'] = pd.Series(pd.NA, index=master.index, dtype='Int64')
master['recipe_precursor'] = pd.Series(pd.NA, index=master.index, dtype='string')
master['phase'] = pd.Series(pd.NA, index=master.index, dtype='string')
master['nominal_dose_time_s'] = np.nan
master['nominal_purge_time_s'] = np.nan
master['ald_process_type'] = pd.Series(pd.NA, index=master.index, dtype='string')

elapsed = master[ELAPSED_COL]
pulse_map = pulse_map.reset_index(drop=True)

# 첫 pulse 전: recipe step 1 initial purge
first_pulse_t = pulse_map.loc[0, 'pulse_start_elapsed_s']
initial_row = execution.iloc[0]
mask = elapsed < first_pulse_t
master.loc[mask, 'recipe_step'] = int(initial_row['Recipe Step'])
master.loc[mask, 'recipe_precursor'] = 'None'
master.loc[mask, 'phase'] = 'initial_N2_purge'
master.loc[mask, 'nominal_purge_time_s'] = initial_row['Purge Time [s]']
master.loc[mask, 'ald_process_type'] = initial_row['ALD Process Type']

# 각 실제 pulse 시작부터 다음 pulse 시작 직전까지 배정
for i, row in pulse_map.iterrows():
    start_t = row['pulse_start_elapsed_s']
    next_t = pulse_map.loc[i + 1, 'pulse_start_elapsed_s'] if i + 1 < len(pulse_map) else np.inf
    step = int(row['Recipe Step'])
    precursor = row['Precursor']
    dose_end = start_t + float(row['Dose Time [s]'])

    # dose는 실제 do 신호로 최종 판정
    dose_mask = (elapsed >= start_t) & (elapsed < next_t) & master['dose_signal'].eq(precursor)
    purge_mask = (elapsed >= start_t) & (elapsed < next_t) & master['dose_signal'].eq('None')

    for m in (dose_mask, purge_mask):
        master.loc[m, 'recipe_step'] = step
        master.loc[m, 'recipe_precursor'] = precursor
        master.loc[m, 'nominal_dose_time_s'] = row['Dose Time [s]']
        master.loc[m, 'nominal_purge_time_s'] = row['Purge Time [s]']
        master.loc[m, 'ald_process_type'] = row['ALD Process Type']

    master.loc[dose_mask, 'phase'] = f'{precursor}_dose'
    master.loc[purge_mask, 'phase'] = f'{precursor}_N2_purge'

    # 현재 pulse step과 다음 pulse step 사이에 독립 purge recipe가 있으면 분리
    next_step = int(pulse_map.loc[i + 1, 'Recipe Step']) if i + 1 < len(pulse_map) else int(execution['Recipe Step'].max()) + 1
    none_steps = execution.loc[
        (execution['Recipe Step'] > step) &
        (execution['Recipe Step'] < next_step) &
        execution['Precursor'].eq('None')
    ]
    if not none_steps.empty:
        boundary = start_t + float(row['Dose Time [s]']) + float(row['Purge Time [s]'])
        for _, none_row in none_steps.iterrows():
            none_end = min(boundary + float(none_row['Purge Time [s]']), next_t)
            none_mask = (elapsed >= boundary) & (elapsed < none_end) & master['dose_signal'].eq('None')
            master.loc[none_mask, 'recipe_step'] = int(none_row['Recipe Step'])
            master.loc[none_mask, 'recipe_precursor'] = 'None'
            master.loc[none_mask, 'phase'] = 'long_N2_purge'
            master.loc[none_mask, 'nominal_dose_time_s'] = np.nan
            master.loc[none_mask, 'nominal_purge_time_s'] = none_row['Purge Time [s]']
            master.loc[none_mask, 'ald_process_type'] = none_row['ALD Process Type']
            boundary = none_end

# 마지막 pulse의 명목 purge 이후: final purge step
final_rows = execution.loc[(execution['Recipe Step'] > int(pulse_map.iloc[-1]['Recipe Step'])) & execution['Precursor'].eq('None')]
if not final_rows.empty:
    last = pulse_map.iloc[-1]
    final_start = last['pulse_start_elapsed_s'] + float(last['Dose Time [s]']) + float(last['Purge Time [s]'])
    final_row = final_rows.iloc[-1]
    final_mask = elapsed >= final_start
    master.loc[final_mask, 'recipe_step'] = int(final_row['Recipe Step'])
    master.loc[final_mask, 'recipe_precursor'] = 'None'
    master.loc[final_mask, 'phase'] = 'final_N2_purge'
    master.loc[final_mask, 'nominal_dose_time_s'] = np.nan
    master.loc[final_mask, 'nominal_purge_time_s'] = final_row['Purge Time [s]']
    master.loc[final_mask, 'ald_process_type'] = final_row['ALD Process Type']

if master['phase'].isna().any():
    raise ValueError(f"phase가 배정되지 않은 행: {master['phase'].isna().sum()}")

display(master['phase'].value_counts().rename_axis('phase').to_frame('rows'))

## 6. Cycle ID 및 분석용 flag 추가

In [ ]:
# TMA pulse 시작마다 새 cycle. 첫 TMA 전은 cycle 0.
master['cycle_id'] = (master['pulse_start'] & master['dose_signal'].eq('TMA')).cumsum().astype(int)
h2o_cycles = set(master.loc[master['phase'].eq('H2O_dose'), 'cycle_id'].unique())
master['is_tma_only_cycle'] = (master['cycle_id'] > 0) & ~master['cycle_id'].isin(h2o_cycles)
master['is_n2_purge'] = master['phase'].str.contains('N2_purge', na=False)
master['xtal_valid'] = True
xtal_col = next((c for c in master.columns if c.startswith('Xtal OK?')), None)
if xtal_col:
    master['xtal_valid'] = pd.to_numeric(master[xtal_col], errors='coerce').eq(1)

summary = pd.DataFrame({
    'metric': ['rows', 'cycles', 'TMA pulses', 'H2O pulses', 'unassigned phases', 'invalid xtal rows'],
    'value': [len(master), master['cycle_id'].max(), (master['phase'] == 'TMA_dose').astype(int).diff().eq(1).sum(), (master['phase'] == 'H2O_dose').astype(int).diff().eq(1).sum(), master['phase'].isna().sum(), (~master['xtal_valid']).sum()]
})
display(summary)
display(master[[TIME_COL, ELAPSED_COL, DO1_COL, DO2_COL, 'recipe_step', 'cycle_id', 'recipe_precursor', 'phase']].head(610).tail(20))

## 7. 단일 master table 저장 및 다운로드

In [ ]:
front = [
    TIME_COL, ELAPSED_COL, 'recipe_step', 'cycle_id', 'recipe_precursor', 'phase',
    'nominal_dose_time_s', 'nominal_purge_time_s', 'ald_process_type',
    DO1_COL, DO2_COL, 'is_n2_purge', 'is_tma_only_cycle', 'xtal_valid'
]
remaining = [c for c in master.columns if c not in front and c not in ('dose_signal', 'pulse_start', 'pulse_id')]
master = master[front + remaining].copy()

OUTPUT = 'integrated_timestamp_recipe_master.csv'
master.to_csv(OUTPUT, index=False, encoding='utf-8-sig')
print(f'저장 완료: {OUTPUT} ({len(master):,} rows × {master.shape[1]} columns)')
display(master.head())

try:
    from google.colab import files
    files.download(OUTPUT)
except ImportError:
    print(Path(OUTPUT).resolve())

## 다음 분석
이 master table을 기준으로 (1) 온도에 따른 QMS/QCM 변화와 (2) N2 purge pressure에 따른 변화를 분석할 수 있습니다. Cycle-level 요약은 별도 원본 파일로 관리하지 않고 분석 시 이 테이블에서 파생합니다.